# Прогноз отгрузок по маршрутам: финальное решение

**Команда:** 6 cults of 7 spirulinas. **Публичная метрика:** 0,3224 (WAPE + |Relative Bias|).

Решение — ансамбль:
1. **Нейросеть GRU** — одна сеть на все 1000 маршрутов, прогноз сразу на 8 шагов (11:00–14:30).
   Обучаются два варианта: `recent` (28 дней истории, свежие примеры весят больше) и `base` (21 день, равные веса).
2. **Градиентный бустинг** — готовый прогноз из ноутбука бустинга (`submission_residual_v8.csv`):
   LightGBM + гребневая регрессия по маршрутам + бустинг на ошибках.
3. **Смесь** 60% GRU + 40% бустинг и **калибровка общего уровня** суммы.

**Что нужно для запуска**
- `data/train_solo_track.parquet`, `data/test_solo_track.parquet` — данные кейса;
- `submission_residual_v8.csv` — прогноз бустинга (рядом с ноутбуком);
- библиотеки: `numpy pandas pyarrow torch`.

На видеокарте ноутбук работает несколько минут, на процессоре — около двух часов.
Обучение нейросети случайно, поэтому итоговый файл совпадает с отправленным не до последнего знака.

## 1. Настройки

In [ ]:
import time
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn

DATA = Path('../data')
BOOSTING_CSV = Path('submission_residual_v8.csv')   # прогноз бустинга
OUT = Path('submission_final.csv')

H, DAY = 8, 48            # горизонт: 8 шагов по 30 минут; 48 шагов в сутках
LOOKBACK = 48             # сеть видит последние сутки
EPOCHS = 6
W_GRU = 0.6               # доля нейросети в смеси
LEVEL = 1.0012            # множитель уровня относительно суммы бустинга (найден пробами на лидерборде)
RUN_VALIDATION = False    # True - проверить нейросеть на 5 последних днях (удваивает время)
FAST = False              # True - быстрый прогон для проверки, что всё запускается

# варианты нейросети: дней истории, период полураспада веса по давности (0 = равные веса), число запусков
GRU_VARIANTS = {'recent': dict(days=28, half_life=7, seeds=2),
                'base':   dict(days=21, half_life=0, seeds=6)}
if FAST:
    EPOCHS = 1
    GRU_VARIANTS = {'recent': dict(days=3, half_life=7, seeds=1), 'base': dict(days=2, half_life=0, seeds=1)}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('устройство:', DEVICE)

## 2. Данные
Таблица разворачивается в матрицы «маршрут × время»: отгрузки `Y` и шесть статусов `S`.

In [ ]:
tr = pd.read_parquet(DATA / 'train_solo_track.parquet').sort_values(['route_id', 'timestamp'])
te = pd.read_parquet(DATA / 'test_solo_track.parquet')
routes = np.sort(tr.route_id.unique()); times = np.sort(tr.timestamp.unique())
R, T = len(routes), len(times)
assert len(tr) == R * T, 'ожидается полная сетка маршрут x время'
Y = tr.target_1h.to_numpy(np.float32).reshape(R, T)
S = np.stack([tr[f'status_{i}'].to_numpy(np.float32).reshape(R, T) for i in range(1, 7)], 0)
del tr

t0 = pd.Timestamp(times[0])
future = np.sort(te.timestamp.unique())
assert len(future) == H and pd.Timestamp(future[0]) == pd.Timestamp(times[-1]) + pd.Timedelta(minutes=30)
start = pd.Timestamp(future[0])
print(f'маршрутов {R}, шагов {T}, обучение до {pd.Timestamp(times[-1])}, прогноз {start} .. {pd.Timestamp(future[-1])}')


def stamp(o):
    return t0 + pd.Timedelta(minutes=30 * int(o))


slot = np.arange(T + H) + t0.hour * 2 + t0.minute // 30
tod = (slot % DAY).astype(np.float32); dow = ((slot // DAY + t0.dayofweek) % 7).astype(np.float32)
TF = np.stack([np.sin(2 * np.pi * tod / DAY), np.cos(2 * np.pi * tod / DAY),
               np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)], 1)
Ye = np.concatenate([Y, np.full((R, H), np.nan, np.float32)], 1)

## 3. Нейросеть
- Вход: последние 48 получасов отгрузок и статусов (нормированы на масштаб маршрута), время суток и день недели,
  обучаемый вектор маршрута; для каждой будущей точки — значения «вчера», «неделю назад» и среднее за 7 дней.
- Архитектура: GRU(96) → 256 → 128 → 8.
- Потери: абсолютная ошибка с весом = масштаб маршрута (соответствует WAPE).

In [ ]:
class Net(nn.Module):
    def __init__(self, n_routes, d_in=11, d_fut=H * 7, hid=96, emb=16):
        super().__init__()
        self.emb = nn.Embedding(n_routes, emb)
        self.gru = nn.GRU(d_in, hid, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid + emb + d_fut, 256), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, H))

    def forward(self, seq, fut, r):
        _, h = self.gru(seq)
        return self.head(torch.cat([h[-1], self.emb(r), fut], 1))


def train_gru(cut, pred_origins, days, half_life, seeds, name=''):
    """Обучает сеть на окнах, целиком лежащих до момента `cut`, и прогнозирует для стартов `pred_origins`.
    Возвращает прогноз (len(pred_origins) * R * H) в порядке: старт -> маршрут -> шаг."""
    lo = max(0, cut - 28 * DAY)
    sy = np.maximum(Y[:, lo:cut].mean(1), 1.0).astype(np.float32)
    ss = np.maximum(S[:, :, lo:cut].mean(2), 1.0).astype(np.float32)
    Yn = Y / sy[:, None]; Sn = np.log1p(S / ss[:, :, None]); Yen = Ye / sy[:, None]

    def batch(rr, oo):
        idx = oo[:, None] + np.arange(-LOOKBACK, 0)[None, :]
        seq = np.concatenate([Yn[rr[:, None], idx][..., None],
                              np.moveaxis(Sn[:, rr[:, None], idx], 0, -1), TF[idx]], -1)
        fidx = oo[:, None] + np.arange(H)[None, :]
        fut = np.concatenate([TF[fidx],
                              np.nan_to_num(Yen[rr[:, None], fidx - DAY])[..., None],
                              np.nan_to_num(Yen[rr[:, None], fidx - 7 * DAY])[..., None],
                              np.nan_to_num(np.mean([Yen[rr[:, None], fidx - d * DAY] for d in range(1, 8)], 0))[..., None]], -1)
        return (torch.tensor(seq, dtype=torch.float32), torch.tensor(fut.reshape(len(rr), -1), dtype=torch.float32),
                torch.tensor(rr, dtype=torch.long))

    o_train = np.arange(max(LOOKBACK + 7 * DAY, cut - days * DAY), cut - H + 1)
    pairs = np.array(np.meshgrid(np.arange(R), o_train)).T.reshape(-1, 2)
    y_all = np.stack([Yn[pairs[:, 0], pairs[:, 1] + h] for h in range(H)], 1).astype(np.float32)
    w_pair = np.ones(len(pairs), np.float32)
    if half_life > 0:
        w_pair *= (0.5 ** ((cut - pairs[:, 1]) / DAY / half_life)).astype(np.float32)
    w_pair /= w_pair.mean()
    print(f'[{name}] примеров {len(pairs)}, дней {days}, полураспад {half_life}, запусков {seeds}', flush=True)

    preds = []
    for seed in range(seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        net = Net(R).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * (len(pairs) // 1024 + 1))
        w_route = torch.tensor(sy / sy.mean(), dtype=torch.float32, device=DEVICE)
        for ep in range(EPOCHS):
            net.train(); perm = np.random.permutation(len(pairs)); tot = 0.0; ts = time.time()
            for b in range(0, len(perm), 1024):
                ib = perm[b:b + 1024]
                seq, fut, r = batch(pairs[ib, 0], pairs[ib, 1])
                seq, fut, r = seq.to(DEVICE), fut.to(DEVICE), r.to(DEVICE)
                yt = torch.tensor(y_all[ib], device=DEVICE); wp = torch.tensor(w_pair[ib], device=DEVICE)
                loss = ((net(seq, fut, r) - yt).abs() * w_route[r][:, None] * wp[:, None]).mean()
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step(); sched.step(); tot += loss.item() * len(ib)
            print(f'  запуск {seed + 1}/{seeds}, эпоха {ep + 1}/{EPOCHS}, потери {tot / len(perm):.4f} ({time.time() - ts:.0f} c)', flush=True)
        net.eval(); out = []
        with torch.no_grad():
            for o in pred_origins:
                seq, fut, r = batch(np.arange(R), np.full(R, int(o)))
                p = net(seq.to(DEVICE), fut.to(DEVICE), r.to(DEVICE)).cpu().numpy()
                out.append((np.maximum(p, 0) * sy[:, None]).ravel())
        preds.append(np.concatenate(out))
    return np.mean(preds, 0)

## 4. Проверка на прошлых днях (по желанию)
Имитация теста: модель видит данные только до 10:30 первого из пяти последних дней и прогнозирует 11:00–14:30
для каждого из них. Показатель — WAPE при правильном уровне суммы (чистая точность прогноза).
Результаты команды: `recent` 0,3155, `base` 0,3160; медиана за 28 дней 0,347.

In [ ]:
if RUN_VALIDATION:
    vv = np.array([o for o in range(DAY * 8, T - H + 1) if stamp(o).time() == start.time()])[-5:]
    act = np.concatenate([Y[:, o:o + H].ravel() for o in vv]); RH = R * H
    for name, cfg in GRU_VARIANTS.items():
        pv = train_gru(int(vv[0]), vv, name=name + ', проверка', **cfg)
        wape = [np.abs(act[i*RH:(i+1)*RH] - pv[i*RH:(i+1)*RH] * act[i*RH:(i+1)*RH].sum() / pv[i*RH:(i+1)*RH].sum()).sum()
                / act[i*RH:(i+1)*RH].sum() for i in range(len(vv))]
        print(f'{name}: WAPE при правильном уровне {np.mean(wape):.4f}, по дням {np.round(wape, 4).tolist()}')

## 5. Обучение на всей истории и прогноз теста

In [ ]:
gru_pred = {name: train_gru(T, np.array([T]), name=name, **cfg) for name, cfg in GRU_VARIANTS.items()}
gru = np.mean(list(gru_pred.values()), 0)
for name, p in gru_pred.items():
    print(f'{name}: сумма прогноза {p.sum() / 1e6:.0f} млн')

## 6. Смесь с бустингом и калибровка уровня
Метрика = WAPE + |Relative Bias|. Второе слагаемое зависит только от общей суммы прогноза,
поэтому сумма выставляется отдельно: сумма бустинга × `LEVEL`.

In [ ]:
lookup = pd.DataFrame({'route_id': np.repeat(routes, H), 'timestamp': np.tile(future, R), 'gru': gru})
m = te[['id', 'route_id', 'timestamp']].merge(lookup, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
boost = pd.read_csv(BOOSTING_CSV).rename(columns={'y_pred': 'boost'})
m = m.merge(boost, on='id', how='left', validate='one_to_one')
assert len(m) == len(te) and m[['gru', 'boost']].notna().all().all(), 'id в прогнозе бустинга не совпадают с тестом'

pred = W_GRU * m['gru'] + (1 - W_GRU) * m['boost']
pred *= m['boost'].sum() * LEVEL / pred.sum()
sub = pd.DataFrame({'id': m['id'], 'y_pred': pred.clip(lower=0)})

assert list(sub.columns) == ['id', 'y_pred'] and len(sub) == len(te)
assert sub.id.is_unique and set(sub.id) == set(te.id)
assert np.isfinite(sub.y_pred).all() and (sub.y_pred >= 0).all()
sub.to_csv(OUT, index=False)
print(f'сохранено: {OUT}, строк {len(sub)}, сумма {sub.y_pred.sum() / 1e6:.1f} млн')
sub.head()